In [1]:
%run nb_dq_utils

StatementMeta(, ace150e6-020d-418b-9f20-6e9adaa33fda, 10, Finished, Available, Finished, True)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import functions as F
 
logger = get_logger("silver.employee.dq")   # was a bespoke logger — now shared

# ── Read Bronze ───────────────────────────────────────────────────────
df_employee = spark.read.table("lh_adventureworks_bronze.dbo.HumanResources_Employee")
df_person = spark.read.table("lh_adventureworks_bronze.dbo.Person_Person")
df_dept = spark.read.table("lh_adventureworks_bronze.dbo.HumanResources_Department")

# Note: EmployeeDepartmentHistory wasn't in original Bronze list —
# checking if it exists; if not, we'll handle department differently
try:
    df_edh = spark.read.table("lh_adventureworks_bronze.dbo.HumanResources_EmployeeDepartmentHistory")
    has_edh = True
except Exception:
    has_edh = False
    logger.info("Note: EmployeeDepartmentHistory not in Bronze — department link will be skipped for now")
expected_rows = df_employee.count() 

logger.info(f"Employee rows  : {expected_rows}")
logger.info(f"Person rows  : {df_person.count()}")
logger.info(f"Department rows  : {df_dept.count()}")
logger.info(f"Has EDH table  : {has_edh}")

StatementMeta(, ace150e6-020d-418b-9f20-6e9adaa33fda, 11, Finished, Available, Finished, False)

INFO:silver.employee.dq:Employee rows  : 290
INFO:silver.employee.dq:Person rows  : 19972
INFO:silver.employee.dq:Department rows  : 16
INFO:silver.employee.dq:Has EDH table  : True


In [3]:
df_person_clean = df_person.select(
    F.col("BusinessEntityID").alias("p_BusinessEntityID"),
    F.col("FirstName"), F.col("MiddleName"), F.col("LastName"), F.col("EmailPromotion")
)
 
df_silver_employee = df_employee.join(
    df_person_clean,
    df_employee["BusinessEntityID"] == df_person_clean["p_BusinessEntityID"],
    how="left"
).drop("p_BusinessEntityID")
 
# ── FIXED: has_edh is now actually used, instead of being set and ignored ──
if has_edh:
    df_edh_current = df_edh.filter(F.col("EndDate").isNull()).select(
        F.col("BusinessEntityID").alias("edh_BusinessEntityID"),
        F.col("DepartmentID"),
        F.col("ShiftID"),
        F.col("StartDate").alias("DepartmentStartDate")
    )
    df_silver_employee = df_silver_employee.join(
        df_edh_current,
        df_silver_employee["BusinessEntityID"] == df_edh_current["edh_BusinessEntityID"],
        how="left"
    ).drop("edh_BusinessEntityID")
else:
    # Keep the schema identical either way, so the final .select() below
    # never breaks regardless of whether EDH was available this run.
    df_silver_employee = df_silver_employee \
        .withColumn("DepartmentID", F.lit(None).cast("integer")) \
        .withColumn("ShiftID", F.lit(None).cast("integer")) \
        .withColumn("DepartmentStartDate", F.lit(None).cast("timestamp"))
 
df_dept_clean = df_dept.select(
    F.col("DepartmentID").alias("d_DepartmentID"),
    F.col("Name").alias("DepartmentName"),
    F.col("GroupName")
)
 
df_silver_employee = df_silver_employee.join(
    df_dept_clean,
    df_silver_employee["DepartmentID"] == df_dept_clean["d_DepartmentID"],
    how="left"
).drop("d_DepartmentID")
 
df_silver_employee = df_silver_employee.withColumn(
    "EmployeeFullName",
    F.concat_ws(" ", F.col("FirstName"), F.col("MiddleName"), F.col("LastName"))
)
 
df_silver_employee = df_silver_employee.select(
    F.col("BusinessEntityID").alias("EmployeeID"),
    "EmployeeFullName", "FirstName", "LastName",
    F.col("NationalIDNumber"), F.col("LoginID"), F.col("OrganizationNode"),
    F.col("OrganizationLevel"), F.col("JobTitle"), F.col("BirthDate"),
    F.col("MaritalStatus"), F.col("Gender"), F.col("HireDate"), F.col("SalariedFlag"),
    F.col("VacationHours"), F.col("SickLeaveHours"), F.col("CurrentFlag"),
    "DepartmentID", "DepartmentName", "GroupName", "ShiftID", "DepartmentStartDate",
    F.col("rowguid"), F.col("ModifiedDate")
)
 
df_silver_employee = df_silver_employee \
    .withColumn("DepartmentID", F.col("DepartmentID").cast("integer")) \
    .withColumn("ShiftID", F.col("ShiftID").cast("integer")) \
    .withColumn("OrganizationLevel", F.col("OrganizationLevel").cast("integer"))
 
# ── Cache before any checks run ─────────────────────────────────────
df_silver_employee.cache()

StatementMeta(, ace150e6-020d-418b-9f20-6e9adaa33fda, 12, Finished, Available, Finished, False)

DataFrame[EmployeeID: int, EmployeeFullName: string, FirstName: string, LastName: string, NationalIDNumber: string, LoginID: string, OrganizationNode: string, OrganizationLevel: int, JobTitle: string, BirthDate: date, MaritalStatus: string, Gender: string, HireDate: date, SalariedFlag: boolean, VacationHours: smallint, SickLeaveHours: smallint, CurrentFlag: boolean, DepartmentID: int, DepartmentName: string, GroupName: string, ShiftID: int, DepartmentStartDate: date, rowguid: string, ModifiedDate: timestamp]

In [4]:
actual_rows = df_silver_employee.count()
 
checks = [
    {
        "name": "Row count",
        "passed": actual_rows == expected_rows,
        "message": f"Expected {expected_rows:,}, got {actual_rows:,}"
    },
    check_null_pk(df_silver_employee, "EmployeeID"),
    check_duplicate_pk(df_silver_employee, "EmployeeID"),
    check_join_coverage(df_silver_employee, "DepartmentID", "DepartmentName",
                         label="Department join coverage"),
]
 
# Kept as a custom check, deliberately — this one is informational-only
# (always passes, never halts the pipeline), and nb_dq_utils doesn't
# currently have a "non-blocking" check type. Not worth adding one for
# a single occurrence; a plain dict here is the right amount of code.
no_dept_at_all = df_silver_employee.filter(F.col("DepartmentID").isNull()).count()
checks.append({
    "name": "Employees with no current department (informational)",
    "passed": True,
    "message": f"{no_dept_at_all:,} employees have no current EDH row (EndDate IS NULL)"
})
 
run_dq_checks(checks, logger, "silver.employee")

StatementMeta(, ace150e6-020d-418b-9f20-6e9adaa33fda, 13, Finished, Available, Finished, False)

INFO:silver.employee.dq:[DQ] [PASS] Row count — Expected 290, got 290
INFO:silver.employee.dq:[DQ] [PASS] Null PK — EmployeeID — 0 nulls
INFO:silver.employee.dq:[DQ] [PASS] Duplicate PK — EmployeeID — 0 duplicates
INFO:silver.employee.dq:[DQ] [PASS] Department join coverage — 0 rows with DepartmentID but no DepartmentName
INFO:silver.employee.dq:[DQ] [PASS] Employees with no current department (informational) — 0 employees have no current EDH row (EndDate IS NULL)
INFO:silver.employee.dq:[DQ] All checks passed for silver.employee.


In [5]:
# ── Write ──────────────────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.hr")
 
df_silver_employee.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_silver.hr.employee")
 
df_verify = spark.read.table("lh_adventureworks_silver.hr.employee")
logger.info(f"silver.hr.employee written — {df_verify.count():,} rows verified.")
 
# ── ADD THIS — release the cached memory now that the write is done ───
df_silver_employee.unpersist()

StatementMeta(, ace150e6-020d-418b-9f20-6e9adaa33fda, 14, Finished, Available, Finished, False)

INFO:silver.employee.dq:silver.hr.employee written — 290 rows verified.


DataFrame[EmployeeID: int, EmployeeFullName: string, FirstName: string, LastName: string, NationalIDNumber: string, LoginID: string, OrganizationNode: string, OrganizationLevel: int, JobTitle: string, BirthDate: date, MaritalStatus: string, Gender: string, HireDate: date, SalariedFlag: boolean, VacationHours: smallint, SickLeaveHours: smallint, CurrentFlag: boolean, DepartmentID: int, DepartmentName: string, GroupName: string, ShiftID: int, DepartmentStartDate: date, rowguid: string, ModifiedDate: timestamp]